# Provider Intelligence POC | structured peers + source-grounded retrieval

**Run in Google Colab:** Upload this `.ipynb` using **File → Upload notebook**, run all cells, and upload the attached `healthcare_provider_mock_2000.csv` when asked. A local Jupyter run automatically uses that filename if it is in the notebook directory or `upload/`.

This is a synthetic demonstration, not actual CareAllies provider performance. Similarity uses standardized **structured** provider features; document search uses a separate text index. Exact numeric answers come from the selected source snapshot. A verified metric dictionary, specialty table, history, intervention log, and enterprise LLM are not attached. The no-LLM demonstration requires only pandas, numpy, scikit-learn, matplotlib, and IPython. No GPU, paid API, or model download is required. Do not upload live PHI to a personal Colab runtime.

**Configuration:** Change `SNAPSHOT_DATE`, `DEMO_NPI`, and peer filters below. Every output retains NPI and snapshot provenance. Empty peer cohorts raise clear errors rather than quietly relaxing eligibility rules.

In [ ]:
# Colab normally includes these packages; install only if your runtime is missing one.
import sys, subprocess, importlib.util
for package, module in [("pandas", "pandas"), ("numpy", "numpy"), ("scikit-learn", "sklearn"), ("matplotlib", "matplotlib")]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

import re, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:  # A minimal local Python runtime can still execute the notebook.
    def display(value):
        print(value.head(12).to_string() if isinstance(value, pd.DataFrame) else value)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

RANDOM_SEED = 42
SNAPSHOT_DATE = None  # None = latest available; e.g. "2025-12-31"
DEMO_NPI = None       # None = choose an eligible provider from this snapshot
MIN_MEMBERSHIP = 20  # pragmatic screening threshold; NOT a statistical precision guarantee
SOURCE_TABLE = "synthetic_provider_performance_csv"
np.random.seed(RANDOM_SEED)
print("pandas", pd.__version__, "scikit-learn", __import__("sklearn").__version__)

## A. Load, normalize, and inspect

`source_to_clean` is the authoritative reversible schema map. Every original column is retained as a cleaned column in the mart; audit/export fields are excluded **only from vectors**. NPIs are read as strings. There is no averaging across dates, pods, or contracts.

In [ ]:
def locate_csv():
    candidates = [Path("healthcare_provider_mock_2000.csv"),
                  Path("upload/healthcare_provider_mock_2000.csv"),
                  Path("/content/healthcare_provider_mock_2000.csv")]
    for path in candidates:
        if path.is_file():
            return path
    try:
        from google.colab import files
        print("Upload healthcare_provider_mock_2000.csv from the supplied attachment")
        uploaded = files.upload()
        matches = [Path(name) for name in uploaded if name.lower().endswith(".csv")]
        if len(matches) != 1:
            raise FileNotFoundError("Upload exactly one provider CSV; received: " + str(list(uploaded)))
        return matches[0]
    except ImportError as exc:
        raise FileNotFoundError("Place healthcare_provider_mock_2000.csv beside this notebook or in upload/") from exc

CSV_PATH = locate_csv()
raw = pd.read_csv(CSV_PATH, dtype={"PCP NPI": "string"})

def clean_header(header):
    # Uniform snake_case: whitespace, punctuation, %, /, periods and parentheses -> _.
    return re.sub(r"_+", "_", re.sub(r"[^a-z0-9]+", "_", str(header).strip().lower())).strip("_")

source_to_clean = {col: clean_header(col) for col in raw.columns}
assert len(source_to_clean) == len(set(source_to_clean.values())), "Header normalization produced a collision"
df = raw.rename(columns=source_to_clean).copy()
data_dictionary = pd.DataFrame([{"source_column": old, "clean_column": new,
                                 "dtype": str(df[new].dtype)} for old, new in source_to_clean.items()])
assert all(" " not in column for column in df.columns)
display(data_dictionary)
print("Normalized example:", {name: source_to_clean[name] for name in ["PCP NPI", "DSNP %", "ADK R12", "Stars Total"]})
print("Original shape:", raw.shape)
print("Original column names:", list(raw.columns))
display(raw.dtypes.rename("source_dtype").to_frame())

In [ ]:
REQUIRED = ["report_date", "pcp_npi", "pcp_name", "operational_market", "membership",
            "member_months", "risk_cms", "dsnp", "new_members", "risk_closure_rate",
            "stars_total", "stars_gap_closure_rate", "attest_completion_rate", "adk_r12",
            "edk_r12", "readmit_rate_r12", "total_expense_pmpm", "mlr"]
missing = sorted(set(REQUIRED) - set(df.columns))
if missing:
    raise ValueError(f"Required source fields absent: {missing}")
df["pcp_npi"] = df["pcp_npi"].astype("string").str.strip()
df["report_date"] = pd.to_datetime(df["report_date"], errors="raise").dt.normalize()
df["run_date"] = pd.to_datetime(df["rundate"], errors="coerce", utc=True) if "rundate" in df else pd.NaT
invalid_npis = df[~df.pcp_npi.str.fullmatch(r"\d{10}").fillna(False)]
if len(invalid_npis):
    raise ValueError(f"{len(invalid_npis)} rows have missing or invalid 10-digit NPIs")
key = ["pcp_npi", "report_date"]
duplicates = df.loc[df.duplicated(key, keep=False), key + ["managing_entity", "reporting_pod"]]
if len(duplicates):
    display(duplicates.head(20))
    raise ValueError("NPI × report date is not unique; declare attribution grain and weighted aggregation rules first")

rate_cols = [c for c in df if c.endswith("rate") or c in ["dsnp", "members_w_triple_wtd", "mlr"]]
numeric = df.select_dtypes(include="number")
range_report = pd.DataFrame({"min": numeric.min(), "median": numeric.median(), "max": numeric.max(),
                             "missing_pct": numeric.isna().mean().mul(100)})
print("Snapshot counts:"); display(df.groupby("report_date").agg(rows=("pcp_npi", "size"), unique_npi=("pcp_npi", "nunique")))
print("All distinct NPIs:", df.pcp_npi.nunique(), "| Two-date NPIs:", df.groupby("pcp_npi").report_date.nunique().eq(2).sum())
print("Duplicate grain keys:", len(duplicates), "| Duplicated names across NPIs:", int(df.groupby("pcp_name").pcp_npi.nunique().gt(1).sum()))
print("Missing fields:"); display(df.isna().sum().loc[lambda s: s.gt(0)].rename("missing_rows").to_frame())
print("Numeric ranges:"); display(range_report)
if len(df) != 2000 or len(raw.columns) != 73:
    warnings.warn("Input differs from reference mock shape 2,000 × 73; check the attachment/version.")
assert (df.membership >= 0).all() and (df.member_months >= 0).all()
assert ((df.new_members >= 0) & (df.new_members <= df.membership)).all()
print("Negative/out-of-range rate-like values (review semantics, not automatic exclusion):",
      {col: int((df[col].dropna() < 0).sum()) for col in rate_cols if pd.api.types.is_numeric_dtype(df[col]) and (df[col].dropna() < 0).any()})
print("These two unevenly spaced dates do not establish continuous trends or post-outreach effects.")

## B. Snapshot mart and optional demographics adapter

One row is one NPI at the selected as-of date **in this mock file only**. A production attribution key may also need market, managing entity, pod, contract, and plan. The optional demographics adapter rejects ambiguous current matches rather than multiplying provider rows.

In [ ]:
as_of = pd.Timestamp(SNAPSHOT_DATE) if SNAPSHOT_DATE else df.report_date.max()
mart = df.loc[df.report_date.eq(as_of)].copy().reset_index(drop=True)
if mart.empty:
    raise ValueError(f"No rows at {as_of.date()}. Available dates: {df.report_date.dt.date.unique()}")
assert mart.pcp_npi.is_unique and mart.report_date.nunique() == 1
mart["source_record_id"] = mart.pcp_npi + "|" + mart.report_date.dt.strftime("%Y-%m-%d")
mart["low_panel_reliability_flag"] = mart.membership.lt(MIN_MEMBERSHIP)
mart["new_member_share"] = mart.new_members.div(mart.membership.replace(0, np.nan))
mart["log_membership"] = np.log1p(mart.membership.clip(lower=0))
mart["log_member_months"] = np.log1p(mart.member_months.clip(lower=0))
mart["log_engagement_count"] = np.log1p(mart.salesforce_engagement_last_60_days.clip(lower=0))
print(f"Mart: {len(mart)} providers at {as_of.date()}; {mart.low_panel_reliability_flag.sum()} below {MIN_MEMBERSHIP} members")
display(mart[["source_record_id", "pcp_npi", "pcp_name", "operational_market", "managing_entity", "reporting_pod", "membership", "member_months", "run_date"]].head())

def attach_demographics(mart, demographics):
    """Accept one effective record per NPI for the selected as-of date; reject one-to-many matches."""
    demo = demographics.rename(columns={c: clean_header(c) for c in demographics.columns}).copy()
    required = {"provider_npi", "primary_specialty", "effective_date"}
    if required - set(demo):
        raise ValueError(f"Demographics missing: {sorted(required - set(demo))}")
    demo["provider_npi"] = demo.provider_npi.astype("string").str.strip()
    demo["effective_date"] = pd.to_datetime(demo.effective_date, errors="raise")
    demo["end_date"] = pd.to_datetime(demo.end_date, errors="coerce") if "end_date" in demo else pd.NaT
    active = demo[(demo.effective_date <= as_of) & (demo.end_date.isna() | (demo.end_date >= as_of))]
    if active.provider_npi.duplicated().any():
        raise ValueError("Multiple active demographics rows per NPI: define taxonomy/practice selection first")
    if not active.provider_npi.str.fullmatch(r"\d{10}").fillna(False).all():
        raise ValueError("Invalid demographics NPI")
    joined = mart.merge(active.drop(columns=["end_date"]), left_on="pcp_npi", right_on="provider_npi", how="left", validate="one_to_one", indicator=True)
    print("Demographics cardinality:", joined._merge.value_counts().to_dict())
    return joined.drop(columns="_merge")

# Example when an approved demographics CSV is available:
# mart = attach_demographics(mart, pd.read_csv("provider_demographics.csv", dtype={"provider_npi": "string"}))
print("Demographics supplied:", "primary_specialty" in mart)
print("Cross-entity/market NPI attribution in full input:",
      {c: int(df.groupby("pcp_npi")[c].nunique().gt(1).sum()) for c in ["operational_market", "managing_entity", "reporting_pod"]})

## C. Versioned feature manifest and lens budgets

Features are numeric and separately weighted by **family**, so a family with more columns does not automatically dominate. `risk_cms` is tentative case-mix context, not provider behavior. Exact source names and leakage controls are recorded in the manifest. Definitions for ADK, Stars, and risk measures are unavailable; these fields stay as uninterpreted source-labeled proxies pending approval.

In [ ]:
FEATURES = {
    "log_membership": ("Membership", "context", "log1p + median impute + robust scale", "Panel-size context; not an event denominator"),
    "dsnp": ("DSNP %", "context", "median impute + robust scale", "Member mix proxy; not complete case-mix adjustment"),
    "new_member_share": ("New Members / Membership", "context", "derived ratio + median impute + robust scale", "Small panels less reliable"),
    "risk_cms": ("Risk CMS", "context", "median impute + robust scale", "Tentative case-mix proxy; definition unavailable"),
    "risk_closure_rate": ("Risk Closure Rate", "risk", "median impute + robust scale", "Outcome leakage if studying risk closure"),
    "attest_completion_rate": ("Attest Completion Rate", "quality", "median impute + robust scale", "Exclude when comparing it as a held-out outcome"),
    "stars_gap_closure_rate": ("Stars Gap Closure Rate", "quality", "median impute + robust scale", "Exclude when studying Stars outcomes"),
    "attest_measure_quality_rate": ("Attest Measure Quality Rate", "quality", "median impute + robust scale", "Exclude when studying quality outcomes"),
    "adk_r12": ("ADK R12", "utilization", "median impute + robust scale", "Definition/denominator unavailable"),
    "edk_r12": ("EDK R12", "utilization", "median impute + robust scale", "Definition/denominator unavailable"),
    "readmit_rate_r12": ("Readmit Rate R12", "utilization", "median impute + robust scale", "Event denominator unavailable"),
    "follow_up_post_discharge_rate_30_days": ("Follow-Up Post Discharge Rate (30 Days)", "utilization", "median impute + robust scale", "Event denominator unavailable"),
    "total_expense_pmpm": ("Total Expense PMPM", "financial", "median impute + robust scale", "Unadjusted; no paired expense components"),
    "mlr": ("MLR", "financial", "median impute + robust scale", "Do not also embed MLR Delta"),
    "log_engagement_count": ("Salesforce Engagement (last 60 days)", "engagement", "log1p + median impute + robust scale", "Count only, not response or success"),
}
LENSES = {
    "population_context": {"context": 1.0},
    "risk_behavior": {"context": 0.50, "risk": 0.50},
    "quality_behavior": {"context": 0.50, "quality": 0.50},
    "utilization_behavior": {"context": 0.50, "utilization": 0.50},
    "financial_behavior": {"context": 0.50, "financial": 0.50},
    "overall_exploratory": {"context": 0.40, "risk": 0.13, "quality": 0.16,
                            "utilization": 0.16, "financial": 0.12, "engagement": 0.03},
}
assert set(FEATURES) <= set(mart)
assert all(abs(sum(b.values()) - 1) < 1e-9 for b in LENSES.values())
manifest = pd.DataFrame([{"clean_feature": c, "source_column": source, "family": family,
                          "transform_missing_policy": transform, "limitations_leakage_risk": caveat,
                          **{f"weight_{lens}": budget.get(family, 0) / sum(x[1] == family for x in FEATURES.values())
                             for lens, budget in LENSES.items()}}
                         for c, (source, family, transform, caveat) in FEATURES.items()])
display(manifest)
print("Held-out Stars outcome columns never enter ANY vector:", sorted(c for c in mart if c.startswith("star_points") or c.startswith("stars_total")))

## D/E. Baseline behavioral vectors, constrained exact neighbors

Fit medians and robust scales on a **single snapshot**. Clip extreme transformed values at ±5. Weighted Euclidean distance is dimensionless but is **not a percentage similarity**. Market, optional plan/specialty, panel-size ratio and case-mix gap are explicit filters. The engine fits exact nearest neighbors on eligible providers for each query.

In [ ]:
feature_cols = list(FEATURES)
imputer = SimpleImputer(strategy="median", keep_empty_features=True)
scaler = RobustScaler()
X_raw = mart[feature_cols].apply(pd.to_numeric, errors="coerce")
all_missing_features = X_raw.columns[X_raw.isna().all()].tolist()
if all_missing_features:
    raise ValueError(f"Cannot fit all-missing features: {all_missing_features}")
X_scaled = np.clip(scaler.fit_transform(imputer.fit_transform(X_raw)), -5.0, 5.0)
if not np.isfinite(X_scaled).all():
    raise ValueError("Non-finite feature value after transformations")
X_scaled = pd.DataFrame(X_scaled, columns=feature_cols, index=mart.index)

def lens_matrix(lens, context_weight=None):
    if lens not in LENSES:
        raise ValueError(f"Lens must be one of {list(LENSES)}")
    budget = LENSES[lens].copy()
    if context_weight is not None:
        if lens == "population_context" or not 0 < context_weight < 1:
            raise ValueError("Choose a behavior lens and context_weight between 0 and 1")
        others = {family: weight for family, weight in budget.items() if family != "context"}
        budget = {"context": context_weight, **{family: (1-context_weight)*w/sum(others.values()) for family, w in others.items()}}
    weights = [np.sqrt(budget.get(FEATURES[col][1], 0) /
                       sum(v[1] == FEATURES[col][1] for v in FEATURES.values())) for col in feature_cols]
    return X_scaled.to_numpy() * np.array(weights), budget

def _index_for_npi(npi):
    selected = mart.index[mart.pcp_npi.eq(str(npi))].tolist()
    if len(selected) != 1:
        raise ValueError(f"NPI {npi} does not uniquely identify a record at {as_of.date()}")
    return selected[0]

def eligible_indices(npi, same_market=True, same_plan=False, panel_ratio=(0.5, 2.0),
                     min_membership=MIN_MEMBERSHIP, max_risk_cms_gap=None, same_specialty=False):
    q = mart.loc[_index_for_npi(npi)]
    if not (0 < panel_ratio[0] <= panel_ratio[1]):
        raise ValueError("Invalid panel ratio bounds")
    selected = mart.pcp_npi.ne(str(npi)) & mart.membership.ge(min_membership)
    if same_market:
        selected &= mart.operational_market.eq(q.operational_market)
    if same_plan:
        selected &= mart.predominant_plan_pbp.eq(q.predominant_plan_pbp)
    if same_specialty:
        if "primary_specialty" not in mart or pd.isna(q.primary_specialty):
            raise ValueError("Specialty constraint needs a validated demographics join")
        selected &= mart.primary_specialty.eq(q.primary_specialty)
    selected &= mart.membership.between(q.membership*panel_ratio[0], q.membership*panel_ratio[1])
    if max_risk_cms_gap is not None:
        selected &= mart.risk_cms.sub(q.risk_cms).abs().le(max_risk_cms_gap)
    return mart.index[selected].to_numpy()

def find_similar_providers(npi, lens="quality_behavior", k=5, peer_filters=None, context_weight=None):
    if not isinstance(k, int) or not 1 <= k <= 50:
        raise ValueError("k must be an integer between 1 and 50")
    peer_filters = peer_filters or {}
    unknown = set(peer_filters) - {"same_market", "same_plan", "panel_ratio", "min_membership", "max_risk_cms_gap", "same_specialty"}
    if unknown:
        raise ValueError(f"Unknown peer filters: {sorted(unknown)}")
    query_idx = _index_for_npi(npi)
    pool = eligible_indices(npi, **peer_filters)
    if len(pool) < k:
        raise ValueError(f"Only {len(pool)} eligible peers for k={k}. Adjust a named peer filter or choose a different NPI.")
    X, budget = lens_matrix(lens, context_weight)
    nn = NearestNeighbors(n_neighbors=k, metric="euclidean", algorithm="brute").fit(X[pool])
    distances, within_pool = nn.kneighbors(X[[query_idx]])
    indices = pool[within_pool[0]]
    results = mart.loc[indices, ["source_record_id", "pcp_npi", "pcp_name", "operational_market", "membership", "member_months", "risk_cms", "stars_total", "stars_gap_closure_rate", "attest_completion_rate", "risk_closure_rate", "adk_r12", "readmit_rate_r12", "total_expense_pmpm"]].copy().reset_index(drop=True)
    active = [c for c in feature_cols if budget.get(FEATURES[c][1], 0) > 0]
    results.insert(0, "rank", np.arange(1, k+1))
    results.insert(1, "euclidean_distance", distances[0].round(4))
    results["top_feature_differences"] = [
        "; ".join(f"{next((old for old, new in source_to_clean.items() if new == c), c)}: {mart.loc[idx,c]:.3g} vs {mart.loc[query_idx,c]:.3g}"
                  for c in X_scaled.loc[idx, active].sub(X_scaled.loc[query_idx, active]).abs().sort_values(ascending=False).head(3).index)
        for idx in indices]
    results.attrs.update({"source_table": SOURCE_TABLE, "report_date": str(as_of.date()),
                          "lens": lens, "family_weights": budget, "peer_filters": peer_filters,
                          "eligible_peer_count": len(pool), "query_npi": str(npi)})
    return results

default_pool = mart.loc[mart.membership.ge(MIN_MEMBERSHIP)].copy()
default_pool["eligible_count"] = default_pool.pcp_npi.map(lambda n: len(eligible_indices(n)))
default_pool = default_pool.loc[default_pool.eligible_count.ge(5)]
if default_pool.empty:
    raise RuntimeError("No provider supports a five-neighbor demonstration under default filters")
if DEMO_NPI is None:
    DEMO_NPI = str(default_pool.iloc[(default_pool.membership-default_pool.membership.median()).abs().argsort().iloc[0]].pcp_npi)
_index_for_npi(DEMO_NPI)
print("Selected synthetic demo NPI:", DEMO_NPI, "| source record:", mart.loc[_index_for_npi(DEMO_NPI), "source_record_id"])
peers = find_similar_providers(DEMO_NPI, "quality_behavior", 5)
display(peers)
print("Query metadata:", peers.attrs)

## F/G. Exact profiles, peer contrasts, and validation

These functions are the deterministic analytics layer. No generative model reads a metric out of narrative text. Rate medians below are descriptive row-level medians, not denominator-weighted rates; numerator/event denominators are unavailable. Output includes literal source column labels and stable NPI/date record IDs.

In [ ]:
ALLOWED_METRICS = {"stars_total", "stars_gap_closure_rate", "attest_completion_rate",
                   "attest_measure_quality_rate", "risk_closure_rate", "adk_r12", "edk_r12",
                   "readmit_rate_r12", "follow_up_post_discharge_rate_30_days", "total_expense_pmpm", "mlr",
                   "membership", "member_months", "risk_cms"}

def resolve_provider(name_or_npi, market=None):
    term = str(name_or_npi).strip()
    subset = mart[mart.pcp_npi.eq(term)] if re.fullmatch(r"\d{10}", term) else mart[mart.pcp_name.str.casefold().eq(term.casefold())]
    if market is not None:
        subset = subset[subset.operational_market.eq(market)]
    if subset.empty:
        return {"status": "not_found", "matches": []}
    records = subset[["pcp_npi", "pcp_name", "operational_market", "source_record_id"]].to_dict("records")
    return {"status": "ambiguous" if len(records)>1 else "resolved", "matches": records}

def get_provider_profile(npi):
    record = mart.loc[_index_for_npi(npi)]
    return {"source_table": SOURCE_TABLE, "report_date": str(as_of.date()), "record_ids": [record.source_record_id],
            "pcp_npi": str(npi), "pcp_name": record.pcp_name, "operational_market": record.operational_market,
            "managing_entity": record.managing_entity, "reporting_pod": record.reporting_pod,
            "metrics": {c: None if pd.isna(record[c]) else float(record[c]) for c in sorted(ALLOWED_METRICS)},
            "metric_definition_version": None,
            "limitations": ["Synthetic source", "Metric business definitions unavailable", "Membership does not replace measure event denominators"]}

def _validated_peers(query_npi, peer_npis):
    ids = [str(n) for n in peer_npis]
    if len(ids) != len(set(ids)) or str(query_npi) in ids or not ids:
        raise ValueError("Supply one or more distinct peer NPIs excluding the target")
    unknown = set(ids) - set(mart.pcp_npi)
    if unknown:
        raise ValueError(f"Peer NPIs not in selected snapshot: {sorted(unknown)}")
    return ids

def compare_provider_to_peers(npi, peer_npis, metric_names):
    ids = _validated_peers(npi, peer_npis)
    fields = list(metric_names)
    if not fields or len(fields) != len(set(fields)) or set(fields) - ALLOWED_METRICS:
        raise ValueError(f"Choose unique allowed metrics: {sorted(ALLOWED_METRICS)}")
    q = mart.loc[_index_for_npi(npi)]
    sub = mart.set_index("pcp_npi").loc[ids]
    rows = []
    for metric in fields:
        values = pd.to_numeric(sub[metric], errors="coerce").dropna()
        rows.append({"clean_metric": metric, "source_column": next((old for old, new in source_to_clean.items() if new == metric), metric),
                     "provider_value": q[metric], "peer_median": values.median() if len(values) else np.nan,
                     "provider_minus_peer_median": q[metric]-values.median() if len(values) else np.nan,
                     "nonmissing_peer_count": len(values)})
    return {"source_table": SOURCE_TABLE, "report_date": str(as_of.date()),
            "record_ids": [q.source_record_id, *sub.source_record_id.tolist()], "sample_size": len(sub),
            "metric_definition_version": None, "comparison": pd.DataFrame(rows),
            "limitations": ["Unadjusted synthetic peer comparison", "Peer row medians are not pooled numerator/denominator rates", "Definitions and event denominators unavailable"]}

def get_peer_group_summary(peer_npis, metric_names):
    ids = [str(n) for n in peer_npis]
    if not ids or len(ids) != len(set(ids)) or set(ids) - set(mart.pcp_npi):
        raise ValueError("All distinct peer NPIs must exist in this snapshot")
    if not metric_names or set(metric_names) - ALLOWED_METRICS:
        raise ValueError("Unsupported metric name")
    subset = mart.set_index("pcp_npi").loc[ids]
    return {"source_table": SOURCE_TABLE, "report_date": str(as_of.date()),
            "record_ids": subset.source_record_id.tolist(), "sample_size": len(ids),
            "summary": subset[list(metric_names)].agg(["count", "median", "min", "max"])}

example = compare_provider_to_peers(DEMO_NPI, peers.pcp_npi.tolist(),
                                    ["risk_closure_rate", "adk_r12", "readmit_rate_r12", "total_expense_pmpm"])
display(example["comparison"])
print({k: v for k, v in example.items() if k != "comparison"})

In [ ]:
# Demo B: hold out ALL quality/Stars variables while matching on member/panel context.
context_peers = find_similar_providers(DEMO_NPI, lens="population_context", k=5,
                                       peer_filters={"same_market": True, "min_membership": MIN_MEMBERSHIP})
stars_comparison = compare_provider_to_peers(DEMO_NPI, context_peers.pcp_npi.tolist(),
                                            ["stars_total", "stars_gap_closure_rate", "attest_completion_rate"])
display(context_peers[["rank", "pcp_npi", "membership", "risk_cms", "stars_total", "top_feature_differences"]])
display(stars_comparison["comparison"])
assert set(FEATURES[c][1] for c in feature_cols if LENSES["population_context"].get(FEATURES[c][1], 0)) == {"context"}
assert not {"stars_total", "star_points_total", "stars_gap_closure_rate", "attest_completion_rate"} & set(
    c for c in feature_cols if LENSES["population_context"].get(FEATURES[c][1], 0))
print("Stars and quality fields were held out of this context-only peer selection. These comparisons do not establish drivers or causation.")

In [ ]:
# Minimal integration checks against the original CSV.
assert len(data_dictionary) == len(raw.columns)
assert df.pcp_npi.str.len().eq(10).all()
assert mart.report_date.eq(as_of).all() and mart.source_record_id.is_unique
original = raw.loc[raw["PCP NPI"].eq(DEMO_NPI) & raw["Report Date"].eq(str(as_of.date()))]
assert len(original) == 1
assert np.isclose(get_provider_profile(DEMO_NPI)["metrics"]["adk_r12"], original.iloc[0]["ADK R12"])
assert np.isclose(example["comparison"].set_index("clean_metric").loc["adk_r12", "peer_median"],
                  mart.set_index("pcp_npi").loc[peers.pcp_npi, "adk_r12"].median())
assert not set(peers.pcp_npi) & {DEMO_NPI}
try:
    find_similar_providers(DEMO_NPI, "population_context", k=5, peer_filters={"same_specialty": True})
except ValueError as exc:
    assert "demographics" in str(exc)
else:
    assert "primary_specialty" in mart  # if a demographics table was explicitly joined
print("PASS: source reconciliation, date isolation, unique IDs, median math, held-out outcomes, missing-demographics guard")

## Sensitivity, segmentation, and visual QA

Neighbor overlap at two context weights is a sensitivity check, not ground truth. PCA is a 2D illustration and distorts original high-dimensional distances. KMeans is optional exploratory segmentation; treat any cluster as descriptive until SME review.

In [ ]:
weighted_30 = find_similar_providers(DEMO_NPI, "quality_behavior", 5, context_weight=0.30)
weighted_70 = find_similar_providers(DEMO_NPI, "quality_behavior", 5, context_weight=0.70)
overlap = len(set(weighted_30.pcp_npi) & set(weighted_70.pcp_npi))
print(f"Top-five peer overlap when context budget changes 0.30 → 0.70: {overlap}/5")
plot_X, _ = lens_matrix("quality_behavior")
pca = PCA(n_components=2, random_state=RANDOM_SEED)
coords = pca.fit_transform(plot_X)
qidx = _index_for_npi(DEMO_NPI)
plt.figure(figsize=(9, 6))
plt.scatter(coords[:,0], coords[:,1], c=mart.membership, cmap="viridis", alpha=.5, s=25)
plt.scatter(coords[qidx,0], coords[qidx,1], color="red", marker="*", s=240, label="Selected NPI")
plt.colorbar(label="Membership")
plt.xlabel("PCA 1"); plt.ylabel("PCA 2")
plt.title(f"Synthetic quality/context vectors, {as_of.date()} | 2D captures {pca.explained_variance_ratio_.sum():.0%} variance")
plt.legend(); plt.tight_layout(); plt.show()

segment_features, _ = lens_matrix("overall_exploratory")
scores = []
for n in (2, 3, 4, 5):
    labels = KMeans(n_clusters=n, random_state=RANDOM_SEED, n_init=10).fit_predict(segment_features)
    scores.append({"clusters": n, "silhouette": silhouette_score(segment_features, labels, sample_size=min(500, len(labels)), random_state=RANDOM_SEED)})
display(pd.DataFrame(scores))
print("No business segment names or quality claims are justified by silhouette scores alone.")

## H. Profile corpus, honest definitions, and local retrieval

Generated profile documents cite **only** CSV fields. Definition stubs expressly say `definition unavailable`. The default retriever is local lexical TF-IDF and never acts as the numeric system of record. An optional semantic embedding index can be enabled later with an approved model and environment; semantic retrieval is a different operation from the structured provider vectors above.

In [ ]:
DEFINITION_FIELDS = ["adk_r12", "edk_r12", "risk_closure_rate", "stars_total", "stars_gap_closure_rate", "mlr"]
def get_metric_definition(metric_name, definition_version=None):
    if metric_name not in ALLOWED_METRICS:
        raise ValueError("Metric is not allowlisted")
    if definition_version is not None:
        raise ValueError("No verified versioned metric dictionary has been supplied")
    source = next((old for old, new in source_to_clean.items() if new == metric_name), metric_name)
    return {"clean_metric": metric_name, "source_label": source, "definition": None,
            "definition_status": "unavailable: no approved metric dictionary attached",
            "definition_version": None, "source": "source CSV header only; project-authored placeholder"}

def build_corpus(snapshot):
    documents = []
    for _, r in snapshot.iterrows():
        metrics = ["membership", "member_months", "risk_cms", "risk_closure_rate",
                   "stars_total", "stars_gap_closure_rate", "adk_r12", "readmit_rate_r12", "total_expense_pmpm"]
        lines = [f"{source_to_clean and next((old for old, new in source_to_clean.items() if new == field), field)}: {r[field]}" for field in metrics]
        documents.append({"doc_id": "profile:" + r.source_record_id,
                          "kind": "profile", "npi": str(r.pcp_npi), "report_date": str(r.report_date.date()),
                          "source_table": SOURCE_TABLE, "record_id": r.source_record_id,
                          "access": "synthetic_demo",
                          "text": f"SYNTHETIC profile NPI {r.pcp_npi}, {r.pcp_name}, {r.operational_market}, snapshot {r.report_date.date()}, record {r.source_record_id}. Source CSV columns:\n" + "\n".join(lines)})
    for field in DEFINITION_FIELDS:
        stub = get_metric_definition(field)
        documents.append({"doc_id": "definition_stub:" + field, "kind": "definition_stub", "npi": None,
                          "report_date": None, "source_table": "project_placeholder_dictionary", "record_id": None,
                          "access": "synthetic_demo", "text": f"Source label: {stub['source_label']}. {stub['definition_status']}. No formula, clinical meaning or official CMS status is asserted."})
    return pd.DataFrame(documents)

corpus = build_corpus(mart)
lexical_vectorizer = TfidfVectorizer(ngram_range=(1,2), min_df=1, strip_accents="unicode")
lexical_index = lexical_vectorizer.fit_transform(corpus.text)

def retrieve_text(query, npi=None, kind=None, k=3, authorization_context="synthetic_demo"):
    if authorization_context != "synthetic_demo":
        raise PermissionError("Only synthetic_demo documents are accessible in this POC")
    allowed = corpus.access.eq(authorization_context)
    if npi is not None:
        _index_for_npi(npi)
        allowed &= corpus.npi.eq(str(npi)) | corpus.kind.eq("definition_stub")
    if kind is not None:
        if kind not in ["profile", "definition_stub"]:
            raise ValueError("Invalid document kind")
        allowed &= corpus.kind.eq(kind)
    allowed_idx = np.flatnonzero(allowed.to_numpy())
    if not len(allowed_idx):
        return []
    similarity = cosine_similarity(lexical_vectorizer.transform([str(query)]), lexical_index[allowed_idx]).ravel()
    ranked = np.argsort(-similarity, kind="stable")[:min(k, len(allowed_idx))]
    return [{**corpus.iloc[allowed_idx[i]][["doc_id", "kind", "npi", "report_date", "source_table", "record_id", "text"]].to_dict(),
             "retrieval_score": round(float(similarity[i]), 4)} for i in ranked if similarity[i] > 0]

print("Indexed", len(corpus), "source-labeled documents, including", len(DEFINITION_FIELDS), "definition stubs")
display(pd.DataFrame(retrieve_text("ADK R12 definition", npi=DEMO_NPI, kind="definition_stub")))

In [ ]:
# OPTIONAL, synthetic-only semantic text embeddings. Enabling this downloads model weights.
# In a controlled enterprise pilot, use only an approved text model and document permissions.
ENABLE_SEMANTIC_TEXT_RETRIEVAL = False
if ENABLE_SEMANTIC_TEXT_RETRIEVAL:
    if importlib.util.find_spec("sentence_transformers") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "sentence-transformers"])
    from sentence_transformers import SentenceTransformer
    text_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    semantic_index = text_model.encode(corpus.text.tolist(), normalize_embeddings=True)

def retrieve_semantic_text(query, npi=None, kind=None, k=3, authorization_context="synthetic_demo"):
    if not ENABLE_SEMANTIC_TEXT_RETRIEVAL:
        raise RuntimeError("Set ENABLE_SEMANTIC_TEXT_RETRIEVAL=True and run the optional model cell first")
    if authorization_context != "synthetic_demo":
        raise PermissionError("Only synthetic_demo documents are accessible")
    allowed = corpus.access.eq(authorization_context)
    if npi is not None:
        _index_for_npi(npi)
        allowed &= corpus.npi.eq(str(npi)) | corpus.kind.eq("definition_stub")
    if kind is not None:
        if kind not in ["profile", "definition_stub"]:
            raise ValueError("Invalid document kind")
        allowed &= corpus.kind.eq(kind)
    candidate_idx = np.flatnonzero(allowed.to_numpy())
    query_vector = text_model.encode([str(query)], normalize_embeddings=True)[0]
    scores = semantic_index[candidate_idx] @ query_vector
    order = np.argsort(-scores)[:min(k, len(candidate_idx))]
    return [{**corpus.iloc[candidate_idx[i]][["doc_id", "kind", "npi", "report_date", "source_table", "record_id", "text"]].to_dict(),
             "retrieval_score": round(float(scores[i]), 4)} for i in order]

# Neither retriever is allowed to invent a metric definition or calculate a numeric peer result.

## I. Tool router and deterministic narrative

The router invokes allowlisted functions for numeric questions; retrieval adds cited source context when relevant. A question requiring outreach history abstains. It does not claim to be a general language model. If an enterprise LLM is later approved, pass these structured tool outputs to a restricted, audited narration adapter and verify each numeric assertion against the tool output.

In [ ]:
def answer_question(question, npi, k=5, peer_filters=None):
    question = str(question).strip()
    if not question or len(question) > 1500:
        raise ValueError("Question must have 1–1500 characters")
    profile = get_provider_profile(npi)
    lower = question.casefold()
    if any(word in lower for word in ["after outreach", "intervention impact", "consistently improved", "respond to outreach"]):
        return {"answer": "Not determinable from these two sparse snapshots and a 60-day engagement count. Dated outreach events, documented interventions and responses, consistent monthly measures and attribution history are required.",
                "record_ids": profile["record_ids"], "retrieved_documents": [], "status": "abstained"}
    stars_intent = "stars" in lower or "higher star" in lower
    quality_intent = "quality" in lower
    lens = ("population_context" if stars_intent else
            "quality_behavior" if quality_intent else
            "utilization_behavior" if "utilization" in lower else "population_context")
    found = find_similar_providers(npi, lens=lens, k=k, peer_filters=peer_filters)
    metrics = (["stars_total", "stars_gap_closure_rate", "attest_completion_rate"] if stars_intent or quality_intent else
               ["adk_r12", "risk_closure_rate", "readmit_rate_r12", "total_expense_pmpm"])
    comparison = compare_provider_to_peers(npi, found.pcp_npi.tolist(), metrics)
    definition_query = "adk_r12" if "adk" in lower else ("stars_total" if stars_intent else None)
    definition = get_metric_definition(definition_query) if definition_query else None
    docs = retrieve_text(question, npi=npi, kind="definition_stub", k=2) if definition_query else []
    lines = [f"Synthetic snapshot {as_of.date()}. NPI {npi} ({profile['pcp_name']}, {profile['operational_market']}).",
             f"Matched {len(found)} peers from {found.attrs['eligible_peer_count']} eligible records using {lens}, filters {found.attrs['peer_filters']}.",
             "Peers, rank / NPI / distance: " + "; ".join(f"{int(r['rank'])} / {r['pcp_npi']} / {r['euclidean_distance']:.3f}" for _, r in found.iterrows()) + "."]
    for _, row in comparison["comparison"].iterrows():
        lines.append(f"{row.source_column}: provider {row.provider_value:.4g}; peer row median {row.peer_median:.4g}; n={int(row.nonmissing_peer_count)}.")
    if definition:
        lines.append(f"{definition['source_label']}: verified business definition unavailable. The numeric comparison does not establish its clinical meaning or official rating status.")
    lines.append("Interpretation: investigate measure definitions, event denominators and case mix before attributing a difference to provider behavior.")
    return {"answer": "\n".join(lines), "status": "answered_with_caveats", "source_table": SOURCE_TABLE,
            "report_date": str(as_of.date()), "record_ids": comparison["record_ids"],
            "metric_definition_version": None, "sample_size": len(found),
            "retrieved_documents": [{"doc_id": d["doc_id"], "report_date": d["report_date"], "source_table": d["source_table"]} for d in docs],
            "peer_table": found, "numeric_comparison": comparison["comparison"]}

for prompt in ["Which providers resemble this PCP on quality behavior?", 
               "Which similar panels have different Stars proxy values?",
               "Compare risk closure, ADK R12, readmission and expense with peers; what is ADK R12?",
               "Which providers consistently improved after outreach over the last 12 months?"]:
    print("\nQUESTION:", prompt)
    response = answer_question(prompt, DEMO_NPI)
    print(response["answer"])
    print("CITATIONS:", response.get("record_ids"), response.get("retrieved_documents"))

## Export for review and next steps

Output files contain synthetic records only. The manifest, mart, neighbor example, and source-labeled corpus can be inspected in Colab's Files pane. Promotion needs a verified metric dictionary, demographic effective dates, event denominators, multi-period source history, authorization rules, SME peer judgments, and enterprise approval before any live data or LLM endpoint is connected.

In [ ]:
OUTPUT = Path("provider_intelligence_outputs")
OUTPUT.mkdir(exist_ok=True)
data_dictionary.to_csv(OUTPUT / "source_to_clean_dictionary.csv", index=False)
manifest.to_csv(OUTPUT / "feature_manifest.csv", index=False)
mart.to_csv(OUTPUT / f"provider_mart_{as_of.date()}.csv", index=False)
peers.to_csv(OUTPUT / "quality_behavior_neighbors.csv", index=False)
corpus.to_json(OUTPUT / "source_labeled_corpus.jsonl", orient="records", lines=True)
print("Wrote:", *[str(p) for p in sorted(OUTPUT.iterdir())], sep="\n- ")
print("Follow-up measures: SME top-k judgment; peer stability across weight/feature changes; documented denominator QA; retrieval citation and abstention checks; time-to-answer vs analyst baseline. No ROI claim from synthetic data.")